# Step 3b: Use the published KKH landslide dataset (Su et al., 2022)

This dataset maps landslides along the Karakoram Highway inside a 10 km buffer, so it fits our study area much better than the NASA catalog.

**Before you run**
1. Download the dataset from Science Data Bank: https://www.doi.org/10.11922/sciencedb.j00001.00374 (a free account may be needed). Check the license and the citation shown on that page.
2. It comes as four `.rar` files. Unpack them with 7-Zip or WinRAR into the folder `data/raw/kkh_landslides/` (keep the sub-folders).
3. Run this notebook from the project folder.

Shapefiles are read with `pyshp`, which does not need GDAL, so it works on your laptop.

## Cell 1: install and settings

In [2]:
%pip install -q pyshp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, glob, json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapefile                      # this is the pyshp library
from pyproj import CRS

DATA_DIR = "files/raw/Landslides points along the KKH"
CORRIDOR_FILE = "files/processed/corridor_10km.geojson"
NORTH = 36.85                         # same northern cut (Khunjerab) as Step 2

print("Working folder:", os.getcwd())
print("Dataset folder exists:", os.path.isdir(DATA_DIR))

Working folder: c:\zaina work pm\kkh-landslide
Dataset folder exists: True


## Cell 2: find the shapefiles
This lists every shapefile found. The one with "point" in its name is the one we use first.

In [4]:
shps = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.shp"), recursive=True))
for s in shps:
    print(s)

points_shp = [s for s in shps if "point" in os.path.basename(s).lower() or "point" in s.lower()]
assert points_shp, "No shapefile with 'point' in its name found. Check Cell 2's list and set points_shp by hand."
PATH = points_shp[0]
print()
print("Using:", PATH)

files/raw/Landslides points along the KKH\Landslides_points_along_the_KKH.shp

Using: files/raw/Landslides points along the KKH\Landslides_points_along_the_KKH.shp


## Cell 3: read it, fix the coordinate system, look at the table

In [5]:
def read_shp(path):
    last_err = None
    for enc in ("utf-8", "gbk", "latin-1"):
        try:
            r = shapefile.Reader(path, encoding=enc, encodingErrors="strict")
            fields = [f[0] for f in r.fields[1:]]
            recs = [list(x) for x in r.records()]
            geoms = [s.__geo_interface__ for s in r.shapes()]
            return fields, recs, geoms
        except Exception as e:
            last_err = e
    raise last_err

fields, recs, geoms = read_shp(PATH)

from shapely.geometry import shape
df = pd.DataFrame(recs, columns=fields)
g = gpd.GeoDataFrame(df, geometry=[shape(x) for x in geoms])

prj = PATH[:-4] + ".prj"
if os.path.exists(prj):
    g = g.set_crs(CRS.from_wkt(open(prj).read()), allow_override=True)
    print("Coordinate system from .prj:", g.crs.to_string()[:60])
else:
    g = g.set_crs("EPSG:4326")
    print("No .prj file, assuming EPSG:4326 (check the coordinates below).")

g = g.to_crs("EPSG:4326")
g.loc[g.geometry.type != "Point", "geometry"] = g.geometry.representative_point()

print("Landslides in the dataset:", len(g))
print("Longitude range:", round(g.geometry.x.min(), 2), "to", round(g.geometry.x.max(), 2))
print("Latitude range:", round(g.geometry.y.min(), 2), "to", round(g.geometry.y.max(), 2))
print()
print("Columns:", list(g.columns))
g.head()

Coordinate system from .prj: EPSG:32643
Landslides in the dataset: 762
Longitude range: 72.88 to 75.57
Latitude range: 34.8 to 38.95

Columns: ['OBJECTID', 'Area', 'ID', 'Aspect', 'Slope', 'Type', 'Longitude', 'Latitude', 'Precipitat', 'Dis_to_Epi', 'Dis_to_Dra', 'Dis_to_KKH', 'Lithology', 'Relief', 'NDVI', 'Dis_to_Fau', 'ORIG_FID', 'geometry']


,OBJECTID,Area,ID,Aspect,Slope,Type,Longitude,Latitude,Precipitat,Dis_to_Epi,Dis_to_Dra,Dis_to_KKH,Lithology,Relief,NDVI,Dis_to_Fau,ORIG_FID,geometry
0,1,1.301426e+05,0,106.638349,28.897624,unstable slope,72.917385,34.913490,807.846233,15226.765625,465.725250,3842.460205,5,920.0,0.262073,8421.881836,0,POINT (72.91724 34.91347)
1,2,2.593022e+05,1,157.338448,37.947968,unstable slope,72.890733,34.906796,808.447306,17549.666016,30.000000,1811.215088,5,1036.0,0.133913,10632.238281,1,POINT (72.89085 34.90682)
2,3,3.262331e+06,2,156.217680,11.258390,unstable slope,72.989177,34.823715,883.424845,10568.348633,30.000000,0.000000,5,277.0,0.086165,12096.251953,2,POINT (72.98893 34.82361)
3,4,8.295825e+05,3,71.857166,25.726408,landslide,72.991650,34.801482,896.856161,13029.259766,1186.802368,1523.220215,5,758.0,0.177938,11838.412109,3,POINT (72.99195 34.80117)
4,5,6.213256e+05,4,145.945533,30.437615,unstable slope,72.979293,34.862596,857.535726,9985.990234,60.000000,1023.962891,5,1160.0,0.171375,10008.001953,4,POINT (72.97895 34.86307)


## Cell 4: what kinds of landslides are in it?
Columns with only a few different values are shown with their counts. Look for a column that describes the landslide type.

In [6]:
for col in g.columns:
    if col == "geometry":
        continue
    n = g[col].nunique(dropna=True)
    if 1 < n <= 12:
        print(g[col].value_counts(dropna=False).head(12))
        print()

Type
unstable slope             412
landslide(debris slide)    167
landslide(rock fall)       126
landslide                   57
Name: count, dtype: int64

Lithology
4    198
1    167
2    141
6    141
5     91
3     24
Name: count, dtype: int64



## Cell 5: count inside our Pakistan corridor, map, save
Two counts: all dataset points south of the Khunjerab cut, and those inside our own 10 km corridor from Step 2.

In [8]:
pak = g[g.geometry.y <= NORTH].copy()
print("Dataset landslides on the Pakistan side (south of the cut):", len(pak))

with open(CORRIDOR_FILE) as f:
    corridor = gpd.GeoDataFrame.from_features(json.load(f)["features"], crs="EPSG:4326").to_crs("EPSG:32643")
area = corridor.union_all() if hasattr(corridor, "union_all") else corridor.unary_union

pak_m = pak.to_crs("EPSG:32643")
inside = pak_m[pak_m.within(area)].copy()
print("Inside our 10 km corridor:", len(inside))

fig, ax = plt.subplots(figsize=(7, 9))
corridor.plot(ax=ax, color="lightblue", edgecolor="steelblue")
pak_m.plot(ax=ax, color="grey", markersize=6, label="dataset, outside our corridor")
if len(inside):
    inside.plot(ax=ax, color="red", markersize=8, label="inside our corridor")
ax.legend()
ax.set_title("Su et al. landslides near the KKH (Pakistan side)")
plt.show()

n = len(inside)
if n >= 100:
    print(f"{n} points: good, enough to train a model.")
elif n >= 30:
    print(f"{n} points: workable.")
else:
    print(f"{n} points: very few. Check the map: the corridor may not match the dataset's road line.")

os.makedirs("data/processed", exist_ok=True)
if n:
    open("data/processed/landslides_su2022_in_corridor.geojson", "w").write(inside.to_json(to_wgs84=True))
    print("Saved data/processed/landslides_su2022_in_corridor.geojson")

Dataset landslides on the Pakistan side (south of the cut): 435


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/corridor_10km.geojson'